# Where Should the 172nd Starbucks Go? Demand-Supply Gap Scoring for Manhattan (Reusable Template)

*A copy-paste framework for identifying over-served and under-served areas in any city*

**Series context:** This is Theme 2B (location fitness) in the Starbucks data science series. See also:
- [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) — EDA & competitor mapping (Theme 0)
- [Starbucks 10-K NLP](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) — Keyword trends, LDA topics (Theme 1)
- [Starbucks Spatial Clustering](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) — Moran's I, LISA, Ripley's K (Theme 2A)
- [Data Pipeline](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) — Full reproducibility pipeline (Notebook C)
- [Walk-Distance Analysis](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) — OSMnx walk-distance analysis (Theme 2C)
- [LDA Topic Explorer](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) — Interactive LDA exploration (Theme 1F)

---

**Context:** [Notebook A](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) showed *where* Starbucks clusters (Moran's I = 0.36, p < 0.001) and confirmed that locations track subway ridership (r = 0.58), not household income (r = 0.03). This notebook asks *why* — and *where next*.

**The question:** Of Manhattan's 309 Census Tracts, which are **over-served** (more stores than demand justifies) and which are **under-served** (demand exists but stores are absent)?

**Method:** We build a **Location Fitness Score (LFS)** = Demand Proxy / Supply Density for each Census Tract, then backtest against known store closures (2017→2026).

## Section 0 — Setup & Data Loading

We load three files from the Manhattan Café Wars dataset:
1. **stores_enriched_v4.csv** — 171 Starbucks with building, transit, competition, and Census features
2. **manhattan_tracts_lisa.geojson** — 309 Census Tract polygons with LISA cluster labels from Notebook A
3. **manhattan_cafes_osm.csv** — 1,335 cafés for computing competitor density per tract

Pedestrian count data (36 Manhattan locations from NYC DOT) is loaded separately for demand estimation.

> **Reuse this template:** Replace the CSVs with your own store + competitor + polygon data.

In [ ]:
!pip install -q geopandas folium plotly osmnx scikit-learn

import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import cKDTree
from pathlib import Path
from IPython.display import display, HTML
import re, warnings
warnings.filterwarnings("ignore")

# ----- Data path auto-detect (Kaggle vs local) -----
KAGGLE_PATHS = [
    Path("/kaggle/input/manhattan-cafe-wars"),
    Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
]
LOCAL_DIR = Path("../../data/processed")
LOCAL_INTERIM = Path("../../data/interim")
LOCAL_RAW = Path("../../data/raw")

DATA_DIR = None
ON_KAGGLE = False
for p in KAGGLE_PATHS:
    if p.exists():
        DATA_DIR = p
        ON_KAGGLE = True
        break
if DATA_DIR is None and LOCAL_DIR.exists():
    DATA_DIR = LOCAL_DIR
if DATA_DIR is None:
    raise FileNotFoundError(
        "Data not found. On Kaggle, add dataset 'shiratoriseto/manhattan-cafe-wars'. "
        "Locally, ensure data/processed/ contains the CSV files."
    )
print(f"Data directory: {DATA_DIR}")

def show_map(m):
    if ON_KAGGLE:
        display(HTML(m._repr_html_()))
    else:
        display(m)

In [ ]:
# ----- Load core datasets -----
df = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")
cafes = pd.read_csv(DATA_DIR / "manhattan_cafes_osm.csv")

# Tract polygons with LISA clusters from Notebook A
GEOJSON_PATH = DATA_DIR / "manhattan_tracts_lisa.geojson"
if not GEOJSON_PATH.exists() and LOCAL_INTERIM.exists():
    GEOJSON_PATH = LOCAL_INTERIM / "manhattan_tracts_lisa.geojson"
tracts_gdf = gpd.read_file(GEOJSON_PATH)

# Pre-computed tract-level demand/supply (from feature engineering step)
TRACT_DS_PATH = DATA_DIR / "tract_demand_supply.csv"
if not TRACT_DS_PATH.exists() and LOCAL_INTERIM.exists():
    TRACT_DS_PATH = LOCAL_INTERIM / "tract_demand_supply.csv"
tract_ds = pd.read_csv(TRACT_DS_PATH, dtype={"GEOID": str})
tract_ds["GEOID"] = tract_ds["GEOID"].str.zfill(11)

print(f"Starbucks stores : {len(df):,} ({df.shape[1]} columns)")
print(f"All cafés        : {len(cafes):,}")
print(f"Census Tracts    : {len(tracts_gdf):,}")

# ----- Load pedestrian count data -----
PED_PATH = DATA_DIR / "manhattan_pedestrian_counts.csv"
if not PED_PATH.exists() and LOCAL_RAW.exists():
    PED_PATH = LOCAL_RAW / "Bi-Annual_Pedestrian_Counts_20260312.csv"

ped_raw = pd.read_csv(PED_PATH)
ped = ped_raw[ped_raw["Borough"] == "Manhattan"].copy()

# Parse coordinates from WKT geometry
def parse_point(geom_str):
    m = re.search(r'POINT\s*\(([-\d.]+)\s+([-\d.]+)\)', str(geom_str))
    return (float(m.group(1)), float(m.group(2))) if m else (np.nan, np.nan)

ped[["ped_lon", "ped_lat"]] = ped["the_geom"].apply(lambda x: pd.Series(parse_point(x)))
ped = ped.dropna(subset=["ped_lat", "ped_lon"]).copy()

# Average recent counts (Oct 2024 + May 2025, AM/PM/Midday)
def safe_int(val):
    try: return int(str(val).replace(",", "").strip())
    except: return np.nan

recent_cols = ["Oct24_AM", "Oct24_PM", "Oct24_MD", "May25_AM", "May25_PM", "May25_MD"]
for col in recent_cols:
    if col in ped.columns:
        ped[col + "_n"] = ped[col].apply(safe_int)
ped["avg_ped_count"] = ped[[c + "_n" for c in recent_cols if c + "_n" in ped.columns]].mean(axis=1)

print(f"Pedestrian count locations (Manhattan): {len(ped)}")
print(f"  Count range: {ped['avg_ped_count'].min():.0f} – {ped['avg_ped_count'].max():.0f}")

# Distance helper
REF_LAT = 40.75
M_PER_DEG_LAT = 111_320
M_PER_DEG_LON = M_PER_DEG_LAT * np.cos(np.radians(REF_LAT))

assert len(df) == 171, f"Expected 171 stores, got {len(df)}"
assert len(tracts_gdf) >= 300, f"Expected ~309 tracts, got {len(tracts_gdf)}"

## Section 1 — The Map of Gaps

Notebook A answered **where** Starbucks clusters. This notebook asks **why** some areas have many stores while others have none — and **where next**.

Of Manhattan's 309 Census Tracts, **205 (66%) have zero Starbucks**. The remaining 104 tracts share all 171 stores. But "no Starbucks" ≠ "opportunity." If demand is low (parks, residential-only blocks), absence is rational. The real question: **are there tracts where demand exists but supply doesn't?**

We answer this with a **Location Fitness Score (LFS)**:

$$\text{LFS} = \frac{\text{Demand Proxy Index}}{\text{Supply Index} + 1}$$

- **High LFS** → under-served (demand outpaces supply → expansion candidate)
- **Low LFS** → over-served (more stores than demand justifies → cannibalization risk)
- **LFS = 0** → no demand signal detected in this tract

The exact thresholds are data-driven — see Section 3 for the full breakdown.

> **Important:** The weights used in this scoring model are assumptions based on Notebook A's correlation findings (ridership *r* = 0.58, income *r* = 0.03), **not calibrated against actual sales data**. We have no revenue figures. Section 4 tests how sensitive the results are to these weight choices, so you can judge the robustness for yourself.

In [ ]:
# ----- Tract-level store counts for the hook visualization -----
# Quick preview: how many Starbucks per tract? (Full LFS computed in Section 3)

# Count Starbucks per tract from stores data
sbux_counts = df.groupby("census_tract_id").size().reset_index(name="n_starbucks")
sbux_counts["census_tract_id"] = sbux_counts["census_tract_id"].astype(str).str.zfill(11)

# Count all cafes per tract via spatial join
cafes_gdf = gpd.GeoDataFrame(
    cafes, geometry=gpd.points_from_xy(cafes.lon, cafes.lat), crs="EPSG:4326"
)
cafes_by_tract = gpd.sjoin(cafes_gdf, tracts_gdf[["GEOID", "geometry"]], how="left", predicate="within")
cafe_counts = cafes_by_tract.groupby("GEOID").size().reset_index(name="n_total_cafes")

# Ensure tract polygons have matching GEOID format
tracts_gdf["GEOID"] = tracts_gdf["GEOID"].astype(str).str.zfill(11)

# Merge counts for preview
tracts_preview = tracts_gdf.copy()
tracts_preview = tracts_preview.merge(sbux_counts, left_on="GEOID", right_on="census_tract_id", how="left")
tracts_preview = tracts_preview.merge(cafe_counts, on="GEOID", how="left")
tracts_preview["n_starbucks"] = tracts_preview["n_starbucks"].fillna(0).astype(int)
tracts_preview["n_total_cafes"] = tracts_preview["n_total_cafes"].fillna(0).astype(int)

n_with = (tracts_preview["n_starbucks"] > 0).sum()
n_without = (tracts_preview["n_starbucks"] == 0).sum()
print(f"Tracts WITH Starbucks:  {n_with} ({n_with/len(tracts_preview)*100:.0f}%)")
print(f"Tracts WITHOUT:         {n_without} ({n_without/len(tracts_preview)*100:.0f}%)")
print(f"Total stores in {n_with} tracts: {tracts_preview['n_starbucks'].sum():.0f}")

# ----- Fig. 1: Starbucks density by tract (preview before LFS) -----
center = [tracts_preview.geometry.centroid.y.mean(), tracts_preview.geometry.centroid.x.mean()]
m1 = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")

folium.Choropleth(
    geo_data=tracts_preview.to_json(),
    data=tracts_preview,
    columns=["GEOID", "n_starbucks"],
    key_on="feature.properties.GEOID",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Starbucks count per Census Tract",
    nan_fill_color="#d9d9d9",
).add_to(m1)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3, color="#00704A", fill=True, fill_opacity=0.8,
        popup=row.get("name", "Starbucks"),
    ).add_to(m1)

m1.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;top:10px;left:60px;z-index:9999;"
    "font-size:14px;font-weight:bold;background:white;padding:6px 12px;"
    "border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,0.3);'>"
    "Fig. 1 — Starbucks Count by Census Tract</div>"
))

show_map(m1)

In [ ]:
# ----- Starbucks distribution by LISA cluster (quick summary) -----
# Which LISA clusters from Notebook A have the most stores?

lisa_summary = tracts_preview.groupby("lisa_cluster").agg(
    tracts=("GEOID", "count"),
    total_starbucks=("n_starbucks", "sum"),
    total_cafes=("n_total_cafes", "sum"),
    avg_starbucks=("n_starbucks", "mean"),
).sort_values("total_starbucks", ascending=False).reset_index()

lisa_summary.columns = ["LISA Cluster", "Tracts", "Starbucks", "All Cafés", "Avg Starbucks/Tract"]

print("Store distribution by LISA spatial cluster (from Notebook A):")
display(lisa_summary.style.format({"Avg Starbucks/Tract": "{:.2f}"}).hide(axis="index"))

print(f"\nKey insight: {n_without} tracts ({n_without/len(tracts_preview)*100:.0f}%) "
      f"have zero Starbucks. But do they have zero *demand*? → Section 2–3 will answer this.")

## Section 2 — Building the Demand Proxy

We cannot observe *actual* coffee demand (no sales data). Instead we build a **Demand Proxy Index (DPI)** from three observable signals, each min-max normalized to [0, 1]:

| Variable | Source | Weight | Rationale |
|---|---|---|---|
| MTA subway ridership | Nearest station's avg daily ridership | **0.50** | Strongest demand signal — *r* = 0.58 with Starbucks density (Notebook A) |
| Pedestrian foot traffic | NYC DOT Bi-Annual Counts (36 locations) | **0.25** | Captures street-level demand not tied to subway |
| Residential population density | ACS 2022 tract population / land area | **0.25** | Morning/evening demand from residents |

$$\text{DPI} = 0.50 \times \hat{R}_{\text{MTA}} + 0.25 \times \hat{P}_{\text{ped}} + 0.25 \times \hat{D}_{\text{pop}}$$

**Why not household income?** Notebook A showed income has *r* = 0.03 with Starbucks density — effectively zero correlation. Including it would add noise, not signal.

> **Reuse tip:** Swap these three variables for your own city's transit/foot-traffic/population data. The framework stays the same.

In [ ]:
# ----- Compute demand proxy components from scratch (for transparency) -----
# This replicates the pre-computed values in tract_demand_supply.csv
# so readers can see exactly how each variable is built.

# 1. Aggregate MTA ridership per tract (sum of all stations in tract)
from shapely.geometry import Point

mta_path = DATA_DIR / "manhattan_mta_ridership_summary.csv"
mta = pd.read_csv(mta_path)
mta_gdf = gpd.GeoDataFrame(mta, geometry=gpd.points_from_xy(mta.lon, mta.lat), crs="EPSG:4326")
mta_by_tract = gpd.sjoin(mta_gdf, tracts_gdf[["GEOID", "geometry"]], how="left", predicate="within")
tract_ridership = mta_by_tract.groupby("GEOID")["avg_daily_ridership"].sum().reset_index()
tract_ridership.columns = ["GEOID", "mta_ridership"]

# 2. Pedestrian counts per tract (assign each counter to its tract)
ped_gdf = gpd.GeoDataFrame(ped, geometry=gpd.points_from_xy(ped.ped_lon, ped.ped_lat), crs="EPSG:4326")
ped_by_tract = gpd.sjoin(ped_gdf, tracts_gdf[["GEOID", "geometry"]], how="left", predicate="within")
tract_ped = ped_by_tract.groupby("GEOID")["avg_ped_count"].mean().reset_index()
tract_ped.columns = ["GEOID", "avg_ped_count"]

# 3. Population density (already in tract_ds, but let's show the calc)
tract_pop = tract_ds[["GEOID", "tract_population", "tract_pop_density"]].copy()

# Merge all into one frame
demand = tracts_gdf[["GEOID", "NAMELSAD", "geometry"]].copy()
demand = demand.merge(tract_ridership, on="GEOID", how="left")
demand = demand.merge(tract_ped, on="GEOID", how="left")
demand = demand.merge(tract_pop, on="GEOID", how="left")
demand[["mta_ridership", "avg_ped_count", "tract_pop_density"]] = \
    demand[["mta_ridership", "avg_ped_count", "tract_pop_density"]].fillna(0)

# Min-max normalize each variable to [0, 1]
for col in ["mta_ridership", "avg_ped_count", "tract_pop_density"]:
    vmin, vmax = demand[col].min(), demand[col].max()
    demand[col + "_norm"] = (demand[col] - vmin) / (vmax - vmin) if vmax > vmin else 0

# Weighted sum → DPI
W_MTA, W_PED, W_POP = 0.50, 0.25, 0.25
demand["dpi"] = (W_MTA * demand["mta_ridership_norm"]
               + W_PED * demand["avg_ped_count_norm"]
               + W_POP * demand["tract_pop_density_norm"])

print("Demand Proxy Index (DPI) — summary statistics:")
print(demand["dpi"].describe().to_string())
print(f"\nTracts with DPI > 0: {(demand['dpi'] > 0).sum()} / {len(demand)}")
print(f"Tracts with MTA data: {(demand['mta_ridership'] > 0).sum()}")
print(f"Tracts with pedestrian data: {(demand['avg_ped_count'] > 0).sum()}")
print(f"Tracts with population data: {(demand['tract_pop_density'] > 0).sum()}")

In [ ]:
# ----- Fig. 2: Histograms of the three demand proxy variables -----
fig2 = make_subplots(rows=1, cols=3,
                     subplot_titles=["MTA Ridership (normalized)",
                                     "Pedestrian Count (normalized)",
                                     "Population Density (normalized)"])

for i, (col, color) in enumerate([
    ("mta_ridership_norm", "#1f77b4"),
    ("avg_ped_count_norm", "#ff7f0e"),
    ("tract_pop_density_norm", "#2ca02c"),
], 1):
    fig2.add_trace(
        go.Histogram(x=demand[col], nbinsx=30, marker_color=color,
                     opacity=0.75, name=col.replace("_norm", "")),
        row=1, col=i,
    )
    fig2.update_xaxes(title_text="Normalized value", row=1, col=i)
    fig2.update_yaxes(title_text="Tracts" if i == 1 else "", row=1, col=i)

fig2.update_layout(
    title="Fig. 2 — Distribution of Demand Proxy Variables (309 Tracts)",
    height=350, showlegend=False,
    template="plotly_white",
)
fig2.show()

In [ ]:
# ----- Fig. 3: Spatial distribution of each demand variable (3 choropleths) -----
fig3_vars = [
    ("mta_ridership_norm", "MTA Ridership", "Blues"),
    ("avg_ped_count_norm", "Pedestrian Count", "Oranges"),
    ("tract_pop_density_norm", "Population Density", "Greens"),
]

for var, label, cmap in fig3_vars:
    m = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")
    folium.Choropleth(
        geo_data=demand.to_json(),
        data=demand,
        columns=["GEOID", var],
        key_on="feature.properties.GEOID",
        fill_color=cmap,
        fill_opacity=0.7,
        line_opacity=0.3,
        legend_name=f"{label} (normalized 0–1)",
        nan_fill_color="#d9d9d9",
    ).add_to(m)
    m.get_root().html.add_child(folium.Element(
        f"<div style='position:fixed;top:10px;left:60px;z-index:9999;"
        f"font-size:14px;font-weight:bold;background:white;padding:6px 12px;"
        f"border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,0.3);'>"
        f"Fig. 3 — {label} by Census Tract</div>"
    ))
    show_map(m)

In [ ]:
# ----- Fig. 4: Correlation matrix of demand proxy variables -----
# Check multicollinearity — if variables are highly correlated (|r| > 0.8),
# weighting them separately adds no information.

corr_cols = ["mta_ridership", "avg_ped_count", "tract_pop_density"]
corr_labels = ["MTA Ridership", "Pedestrian Count", "Pop. Density"]
corr_matrix = demand[corr_cols].corr()

fig4 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_labels, y=corr_labels,
    text=corr_matrix.values.round(2),
    texttemplate="%{text}",
    textfont={"size": 14},
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
))
fig4.update_layout(
    title="Fig. 4 — Correlation Between Demand Proxy Variables",
    height=400, width=500,
    template="plotly_white",
)
fig4.show()

# Print interpretation
max_corr = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
).stack().abs().max()
print(f"\nMax pairwise |r| = {max_corr:.2f}")
if max_corr < 0.8:
    print("✓ No multicollinearity detected (all |r| < 0.8). Safe to combine all three variables.")
else:
    print("⚠ High correlation detected — consider dropping one variable.")

### IDW Interpolation: From 29 Tracts to 309

A critical limitation of the raw pedestrian count data: **only 29 of 309 tracts** contain a sensor. The remaining 280 tracts get `avg_ped_count = 0`, which is not "zero foot traffic" but rather "no measurement."

We use **Inverse Distance Weighting (IDW)** to estimate pedestrian counts at every tract centroid based on nearby sensors:

$$\hat{z}(x_0) = \frac{\sum_{i=1}^{n} w_i \cdot z_i}{\sum_{i=1}^{n} w_i}, \quad w_i = \frac{1}{d(x_0, x_i)^p}$$

where $p = 2$ (standard choice) and $d$ is Euclidean distance in meters.

**Why IDW?** Pedestrian foot traffic is inherently spatial — a tract 200m from a busy counter likely has similar traffic. IDW is simple, interpretable, and requires no parameter tuning beyond the power $p$.

In [ ]:
# ----- IDW interpolation of pedestrian counts to all tract centroids -----
from scipy.spatial import cKDTree

# Sensor locations and values
ped_xy = np.column_stack([
    ped["ped_lon"].values * M_PER_DEG_LON,
    ped["ped_lat"].values * M_PER_DEG_LAT,
])
ped_values = ped["avg_ped_count"].values

# Tract centroids
centroids = demand.geometry.centroid
tract_xy = np.column_stack([
    centroids.x.values * M_PER_DEG_LON,
    centroids.y.values * M_PER_DEG_LAT,
])

# IDW with power=2, using all sensors (36 is small enough)
def idw_interpolate(known_xy, known_values, target_xy, power=2):
    """Inverse Distance Weighting interpolation."""
    tree = cKDTree(known_xy)
    dists, idxs = tree.query(target_xy, k=len(known_xy))
    
    # Handle zero distance (target == sensor location)
    results = np.zeros(len(target_xy))
    for i in range(len(target_xy)):
        d = dists[i]
        v = known_values[idxs[i]]
        zero_mask = d == 0
        if zero_mask.any():
            results[i] = v[zero_mask].mean()
        else:
            w = 1.0 / d**power
            results[i] = np.sum(w * v) / np.sum(w)
    return results

# Compute IDW estimates
demand["ped_count_idw"] = idw_interpolate(ped_xy, ped_values, tract_xy)

# Normalize IDW values to [0,1]
vmin, vmax = demand["ped_count_idw"].min(), demand["ped_count_idw"].max()
demand["ped_count_idw_norm"] = (demand["ped_count_idw"] - vmin) / (vmax - vmin)

# Recompute DPI with IDW pedestrian data
demand["dpi_idw"] = (
    W_MTA * demand["mta_ridership_norm"]
    + W_PED * demand["ped_count_idw_norm"]
    + W_POP * demand["tract_pop_density_norm"]
)

# Compare
print("Pedestrian count coverage:")
print(f"  Raw sensors (point-in-tract): {(demand['avg_ped_count'] > 0).sum()} / {len(demand)} tracts")
print(f"  After IDW interpolation:      {(demand['ped_count_idw'] > 0).sum()} / {len(demand)} tracts")
print(f"\nDPI comparison (original vs IDW-enhanced):")
print(f"  Original — tracts with DPI > 0:  {(demand['dpi'] > 0).sum()}")
print(f"  IDW      — tracts with DPI > 0:  {(demand['dpi_idw'] > 0).sum()}")
print(f"  Correlation between DPI variants: r = {demand['dpi'].corr(demand['dpi_idw']):.3f}")
print(f"\n  IDW pedestrian estimate range: {demand['ped_count_idw'].min():.0f} – {demand['ped_count_idw'].max():.0f}")

In [ ]:
# ----- Fig. 4b: IDW interpolation effect — scatter + choropleth -----

# Scatter: raw vs IDW pedestrian counts per tract
fig_compare = make_subplots(rows=1, cols=2,
    subplot_titles=["Raw vs IDW Pedestrian Counts",
                    "DPI: Original vs IDW-Enhanced"])

fig_compare.add_trace(go.Scatter(
    x=demand["avg_ped_count"], y=demand["ped_count_idw"],
    mode="markers", marker=dict(color="#ff7f0e", opacity=0.5, size=5),
    showlegend=False,
), row=1, col=1)

fig_compare.add_trace(go.Scatter(
    x=demand["dpi"], y=demand["dpi_idw"],
    mode="markers", marker=dict(color="#636efa", opacity=0.5, size=5),
    showlegend=False,
), row=1, col=2)

# Add identity lines
for col_idx in [1, 2]:
    fig_compare.add_shape(type="line", x0=0, x1=1, y0=0, y1=1,
        line=dict(dash="dot", color="grey"), row=1, col=col_idx)

fig_compare.update_xaxes(title_text="Raw Ped Count", row=1, col=1)
fig_compare.update_yaxes(title_text="IDW Ped Count", row=1, col=1)
fig_compare.update_xaxes(title_text="DPI (original)", row=1, col=2)
fig_compare.update_yaxes(title_text="DPI (IDW-enhanced)", row=1, col=2)

r_ped = demand['avg_ped_count'].corr(demand['ped_count_idw'])
r_dpi = demand['dpi'].corr(demand['dpi_idw'])

fig_compare.update_layout(
    title_text=f"Fig. 4b — IDW Fills Data Gaps (ped r={r_ped:.2f}, DPI r={r_dpi:.2f})",
    height=400, width=900, template="plotly_white",
)
fig_compare.show()

# Choropleth: IDW pedestrian estimates
m_idw = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=demand.to_json(),
    data=demand,
    columns=["GEOID", "ped_count_idw"],
    key_on="feature.properties.GEOID",
    fill_color="OrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="IDW Pedestrian Estimate",
).add_to(m_idw)
show_map(m_idw)

print(f"\n★ IDW shifts {(demand['dpi_idw'] > demand['dpi']).sum()} tracts upward")
print(f"  (tracts near busy corridors that previously scored zero)")
print(f"  Max DPI shift: {(demand['dpi_idw'] - demand['dpi']).max():.4f}")

**IDW impact:** Interpolation fills the pedestrian data gap without changing the overall ranking structure (high correlation between original and IDW-enhanced DPI). The main benefit is for tracts *near* busy pedestrian corridors that previously scored zero simply because no sensor fell within their boundary.

> **Caveat:** IDW assumes smooth spatial variation. Sharp breaks (e.g., a highway separating a busy corridor from a residential block) are not captured. For production use, kriging or network-based interpolation would be more appropriate.

We proceed with the **IDW-enhanced DPI** for the Location Fitness Score computation below.

## Section 3 — Location Fitness Score

Now we combine demand and supply into a single score:

$$\text{LFS} = \frac{\text{Demand Proxy Index}}{\text{Supply Index} + 1}$$

**Supply Index** counts the café presence in each tract, weighting the target brand more heavily than competitors:

$$\text{Supply Index} = N_{\text{Starbucks}} + 0.5 \times N_{\text{competitor cafés}}$$

The **+ 1** in the denominator avoids division by zero for tracts with no stores, while keeping the score interpretable: a tract with zero stores and DPI = 0.5 gets LFS = 0.5 / 1 = 0.5, not infinity.

| LFS Range | Interpretation | Action |
|---|---|---|
| **> 0.15** | Under-served — demand exceeds supply | Expansion candidate |
| **0.05 – 0.15** | Roughly balanced | Monitor |
| **< 0.05** | Over-served — supply exceeds demand | Cannibalization risk |

> **Note:** The exact thresholds above are descriptive, not prescriptive. They reflect the distribution of scores in this dataset. The competitor weight (0.5) is also an assumption — Section 4 tests how changing it affects the rankings.

> **Reuse tip:** The `compute_lfs()` function below takes any polygon GeoDataFrame + demand columns + weights. Change the column names and it works for your city.

In [ ]:
# =====================================================================
# compute_lfs() — Reusable Location Fitness Score function
# =====================================================================
# Copy this function into your own notebook. Change the column names
# and demand_weights dict to match your data.
# =====================================================================

def compute_lfs(polygon_gdf, demand_cols, demand_weights,
                brand_col="n_starbucks", competitor_col="n_competitor_cafes",
                competitor_weight=0.5, id_col="GEOID"):
    """
    Compute Location Fitness Score for each polygon (Census Tract, ZIP, etc.).

    Parameters
    ----------
    polygon_gdf : GeoDataFrame
        Must contain columns listed in demand_cols, brand_col, competitor_col.
    demand_cols : list of str
        Raw demand variable column names (will be min-max normalized internally).
    demand_weights : dict {col_name: weight}
        Weights for each demand variable. Must sum to 1.0.
    brand_col : str
        Column with target brand store count per polygon.
    competitor_col : str
        Column with competitor store count per polygon.
    competitor_weight : float
        How much competitors count toward supply (0.5 = half weight).
    id_col : str
        Polygon identifier column.

    Returns
    -------
    GeoDataFrame with added columns:
        - {col}_norm : normalized demand variables
        - demand_proxy_index : weighted sum of normalized demands
        - supply_index : brand_count + competitor_weight * competitor_count
        - lfs : Location Fitness Score
    """
    result = polygon_gdf.copy()

    # --- Validate weights ---
    w_sum = sum(demand_weights.values())
    assert abs(w_sum - 1.0) < 1e-6, f"Weights must sum to 1.0, got {w_sum}"

    # --- Min-max normalize each demand variable ---
    for col in demand_cols:
        vmin, vmax = result[col].min(), result[col].max()
        if vmax > vmin:
            result[col + "_norm"] = (result[col] - vmin) / (vmax - vmin)
        else:
            result[col + "_norm"] = 0.0

    # --- Demand Proxy Index (weighted sum) ---
    result["demand_proxy_index"] = sum(
        demand_weights[col] * result[col + "_norm"] for col in demand_cols
    )

    # --- Supply Index ---
    result["supply_index"] = result[brand_col] + competitor_weight * result[competitor_col]

    # --- LFS = DPI / (Supply + 1) ---
    result["lfs"] = result["demand_proxy_index"] / (result["supply_index"] + 1)

    return result


# =====================================================================
# Apply to Manhattan data
# =====================================================================

# Build the input dataframe: merge demand variables + store counts into tracts
lfs_input = demand[["GEOID", "NAMELSAD", "geometry",
                     "mta_ridership", "avg_ped_count", "ped_count_idw", "tract_pop_density"]].copy()

# Add store counts
lfs_input = lfs_input.merge(
    tracts_preview[["GEOID", "n_starbucks", "n_total_cafes", "lisa_cluster"]],
    on="GEOID", how="left"
)
lfs_input["n_starbucks"] = lfs_input["n_starbucks"].fillna(0).astype(int)
lfs_input["n_total_cafes"] = lfs_input["n_total_cafes"].fillna(0).astype(int)
lfs_input["n_competitor_cafes"] = lfs_input["n_total_cafes"] - lfs_input["n_starbucks"]

# Define demand variables and weights
DEMAND_COLS = ["mta_ridership", "ped_count_idw", "tract_pop_density"]
DEMAND_WEIGHTS = {"mta_ridership": 0.50, "ped_count_idw": 0.25, "tract_pop_density": 0.25}
COMPETITOR_WEIGHT = 0.5

# Compute LFS
tracts_scored = compute_lfs(
    lfs_input,
    demand_cols=DEMAND_COLS,
    demand_weights=DEMAND_WEIGHTS,
    brand_col="n_starbucks",
    competitor_col="n_competitor_cafes",
    competitor_weight=COMPETITOR_WEIGHT,
)

# --- Sanity checks ---
print("LFS computation complete.")
print(f"  Tracts: {len(tracts_scored)}")
print(f"  DPI range:    {tracts_scored['demand_proxy_index'].min():.4f} – {tracts_scored['demand_proxy_index'].max():.4f}")
print(f"  Supply range: {tracts_scored['supply_index'].min():.1f} – {tracts_scored['supply_index'].max():.1f}")
print(f"  LFS range:    {tracts_scored['lfs'].min():.4f} – {tracts_scored['lfs'].max():.4f}")
print(f"  LFS median:   {tracts_scored['lfs'].median():.4f}")
print(f"  LFS mean:     {tracts_scored['lfs'].mean():.4f}")

In [ ]:
# ----- Fig. 5: LFS distribution histogram -----
fig5 = go.Figure()
fig5.add_trace(go.Histogram(
    x=tracts_scored["lfs"], nbinsx=50,
    marker_color="#636efa", opacity=0.75,
))
fig5.add_vline(x=tracts_scored["lfs"].median(), line_dash="dash", line_color="red",
               annotation_text=f"median = {tracts_scored['lfs'].median():.3f}",
               annotation_position="top right")
fig5.update_layout(
    title="Fig. 5 — Distribution of Location Fitness Scores (309 Tracts)",
    xaxis_title="LFS = DPI / (Supply + 1)",
    yaxis_title="Number of Tracts",
    height=350, template="plotly_white",
)
fig5.show()

# Summary stats
n_zero_dpi = (tracts_scored["demand_proxy_index"] == 0).sum()
n_zero_supply = (tracts_scored["supply_index"] == 0).sum()
print(f"Tracts with DPI = 0 (no demand signal): {n_zero_dpi}")
print(f"Tracts with Supply = 0 (no cafés at all): {n_zero_supply}")
print(f"Tracts with LFS = 0: {(tracts_scored['lfs'] == 0).sum()}")

In [ ]:
# ----- Ranking tables: Top 10 under-served & Top 10 over-served -----

rank_cols = ["GEOID", "NAMELSAD", "n_starbucks", "n_competitor_cafes",
             "demand_proxy_index", "supply_index", "lfs", "lisa_cluster"]
display_cols = ["GEOID", "Tract", "Starbucks", "Competitors",
                "Demand (DPI)", "Supply", "LFS", "LISA"]

fmt = {"Demand (DPI)": "{:.3f}", "Supply": "{:.1f}", "LFS": "{:.4f}"}

# --- Under-served: highest LFS (demand present, supply low) ---
# Exclude tracts with DPI = 0 (no demand signal = not a real opportunity)
has_demand = tracts_scored[tracts_scored["demand_proxy_index"] > 0]
top10_under = has_demand.nlargest(10, "lfs")[rank_cols].copy()
top10_under.columns = display_cols

print("=" * 80)
print("TOP 10 UNDER-SERVED TRACTS — Expansion Candidates")
print("  (highest LFS among tracts with demand > 0)")
print("=" * 80)
display(top10_under.style.format(fmt).hide(axis="index").background_gradient(
    subset=["LFS"], cmap="Blues"
))

# --- Over-served: lowest LFS among tracts WITH at least 1 Starbucks ---
has_sbux = tracts_scored[tracts_scored["n_starbucks"] > 0]
top10_over = has_sbux.nsmallest(10, "lfs")[rank_cols].copy()
top10_over.columns = display_cols

print()
print("=" * 80)
print("TOP 10 OVER-SERVED TRACTS — Cannibalization Risk")
print("  (lowest LFS among tracts with ≥ 1 Starbucks)")
print("=" * 80)
display(top10_over.style.format(fmt).hide(axis="index").background_gradient(
    subset=["LFS"], cmap="Reds_r"
))

In [ ]:
# ----- Fig. 6: LFS Choropleth with top candidates highlighted -----
m6 = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")

# Choropleth: all tracts colored by LFS
folium.Choropleth(
    geo_data=tracts_scored.to_json(),
    data=tracts_scored,
    columns=["GEOID", "lfs"],
    key_on="feature.properties.GEOID",
    fill_color="RdYlBu",
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name="Location Fitness Score (blue = under-served, red = over-served)",
    nan_fill_color="#d9d9d9",
).add_to(m6)

# Highlight top 10 under-served with blue markers + labels
top10_ids = set(top10_under["GEOID"].values)
for _, row in tracts_scored.iterrows():
    if row["GEOID"] in top10_ids:
        centroid = row.geometry.centroid
        rank = list(top10_ids).index(row["GEOID"]) + 1 if row["GEOID"] in top10_ids else ""
        folium.Marker(
            location=[centroid.y, centroid.x],
            icon=folium.DivIcon(html=(
                f"<div style='font-size:11px;font-weight:bold;color:white;"
                f"background:#1f77b4;border-radius:50%;width:22px;height:22px;"
                f"text-align:center;line-height:22px;'>"
                f"★</div>"
            )),
            popup=folium.Popup(
                f"<b>{row['NAMELSAD']}</b><br>"
                f"LFS: {row['lfs']:.4f}<br>"
                f"Starbucks: {row['n_starbucks']}<br>"
                f"Competitors: {row['n_competitor_cafes']}<br>"
                f"DPI: {row['demand_proxy_index']:.3f}",
                max_width=200,
            ),
        ).add_to(m6)

# Existing Starbucks as green dots
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3, color="#00704A", fill=True, fill_opacity=0.7,
    ).add_to(m6)

m6.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;top:10px;left:60px;z-index:9999;"
    "font-size:14px;font-weight:bold;background:white;padding:6px 12px;"
    "border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,0.3);'>"
    "Fig. 6 — LFS by Census Tract (★ = top 10 under-served candidates)</div>"
))

show_map(m6)

## Section 4 — Sensitivity Analysis: Do the Weights Matter?

The demand weights and competitor weight in Section 3 are **assumptions**, not calibrated parameters. How much do the rankings change if we use different weights?

We test **two dimensions of sensitivity**:

### A. Demand weight scenarios

| Scenario | MTA | Pedestrian | Population | Rationale |
|---|---|---|---|---|
| **Baseline** | 0.50 | 0.25 | 0.25 | Balanced — our default |
| **Transit-Heavy** | 0.80 | 0.10 | 0.10 | Commuter-focused locations |
| **Residential** | 0.25 | 0.25 | 0.50 | Neighborhood / residential demand |

### B. Competitor weight in Supply Index

| Setting | Formula | Rationale |
|---|---|---|
| **Ignore competitors** | Supply = Starbucks only | Only self-cannibalization matters |
| **Half weight** (baseline) | Supply = Starbucks + 0.5 × competitors | Competitors partially substitute |
| **Full weight** | Supply = Starbucks + 1.0 × competitors | All cafés are equal substitutes |

**What we're looking for:**
- If the same tracts appear in the top 10 across all scenarios → **robust** candidates
- If rankings shuffle dramatically → the model is **weight-sensitive** and conclusions should be qualified
- Spearman rank correlation ρ ≥ 0.8 between scenarios indicates robust rankings

In [ ]:
# =====================================================================
# Part A — Demand weight sensitivity (3 scenarios)
# =====================================================================
from scipy.stats import spearmanr

SCENARIOS = {
    "Baseline\n(0.50/0.25/0.25)": {"mta_ridership": 0.50, "ped_count_idw": 0.25, "tract_pop_density": 0.25},
    "Transit-Heavy\n(0.80/0.10/0.10)": {"mta_ridership": 0.80, "ped_count_idw": 0.10, "tract_pop_density": 0.10},
    "Residential\n(0.25/0.25/0.50)": {"mta_ridership": 0.25, "ped_count_idw": 0.25, "tract_pop_density": 0.50},
}

# Run compute_lfs for each scenario
scenario_results = {}
for name, weights in SCENARIOS.items():
    scored = compute_lfs(
        lfs_input,
        demand_cols=DEMAND_COLS,
        demand_weights=weights,
        brand_col="n_starbucks",
        competitor_col="n_competitor_cafes",
        competitor_weight=COMPETITOR_WEIGHT,
    )
    scenario_results[name] = scored[["GEOID", "NAMELSAD", "lfs", "demand_proxy_index",
                                      "n_starbucks", "n_competitor_cafes"]].copy()
    scenario_results[name] = scenario_results[name].rename(columns={"lfs": f"lfs"})

# Build comparison dataframe: GEOID + LFS for each scenario
compare = lfs_input[["GEOID", "NAMELSAD"]].copy()
for name, res in scenario_results.items():
    compare = compare.merge(
        res[["GEOID", "lfs"]].rename(columns={"lfs": name}),
        on="GEOID", how="left"
    )

# Spearman rank correlation between scenarios
scenario_names = list(SCENARIOS.keys())
n_sc = len(scenario_names)
rho_matrix = np.ones((n_sc, n_sc))
for i in range(n_sc):
    for j in range(i + 1, n_sc):
        rho, _ = spearmanr(compare[scenario_names[i]], compare[scenario_names[j]])
        rho_matrix[i, j] = rho
        rho_matrix[j, i] = rho

# Display short labels for readability
short_labels = ["Baseline", "Transit-Heavy", "Residential"]

fig7 = go.Figure(data=go.Heatmap(
    z=rho_matrix,
    x=short_labels, y=short_labels,
    text=rho_matrix.round(3),
    texttemplate="%{text}",
    textfont={"size": 15},
    colorscale="Blues", zmin=0.5, zmax=1.0,
))
fig7.update_layout(
    title="Fig. 7 — Spearman Rank Correlation Between Demand Weight Scenarios",
    height=400, width=500,
    template="plotly_white",
)
fig7.show()

# Print interpretation
min_rho = rho_matrix[np.triu_indices(n_sc, k=1)].min()
print(f"Minimum pairwise Spearman ρ = {min_rho:.3f}")
if min_rho >= 0.8:
    print("✓ Rankings are ROBUST to demand weight changes (all ρ ≥ 0.8).")
elif min_rho >= 0.6:
    print("△ Rankings are MODERATELY sensitive to demand weight changes.")
else:
    print("✗ Rankings are HIGHLY sensitive — weight choice matters a lot.")

In [ ]:
# ----- Top 10 under-served: which tracts appear in ALL scenarios? -----
TOP_N = 10

top_sets = {}
for name in scenario_names:
    has_demand_sc = compare[compare[name] > 0]  # exclude DPI=0 tracts
    top_ids = set(has_demand_sc.nlargest(TOP_N, name)["GEOID"].values)
    top_sets[name] = top_ids

# Stable candidates: appear in ALL 3 scenarios
stable = set.intersection(*top_sets.values())
# Unstable: appear in exactly 1 scenario
all_top = set.union(*top_sets.values())
in_one_only = {g for g in all_top if sum(g in s for s in top_sets.values()) == 1}

# Build comparison table
rows = []
for geoid in sorted(all_top):
    row = {"GEOID": geoid}
    tract_name = compare.loc[compare["GEOID"] == geoid, "NAMELSAD"].values[0]
    row["Tract"] = tract_name
    for i, name in enumerate(scenario_names):
        lfs_val = compare.loc[compare["GEOID"] == geoid, name].values[0]
        row[short_labels[i]] = lfs_val
        # Rank within this scenario (among tracts with demand > 0)
        has_d = compare[compare[name] > 0]
        rank = (has_d[name] > lfs_val).sum() + 1
        row[f"Rank_{short_labels[i]}"] = rank
    appearances = sum(geoid in s for s in top_sets.values())
    row["In N scenarios"] = appearances
    row["Stability"] = "★ Stable" if geoid in stable else ("Unstable" if geoid in in_one_only else "Partial")
    rows.append(row)

stability_df = pd.DataFrame(rows).sort_values("In N scenarios", ascending=False)

print(f"Unique tracts appearing in ANY top-{TOP_N}: {len(all_top)}")
print(f"★ Stable (in ALL 3 scenarios):             {len(stable)}")
print(f"  Partial (in 2 scenarios):                 {len(all_top) - len(stable) - len(in_one_only)}")
print(f"  Unstable (in 1 scenario only):            {len(in_one_only)}")
print()

def highlight_stability(val):
    if val == "★ Stable":
        return "background-color: #d4edda; font-weight: bold"
    elif val == "Unstable":
        return "background-color: #f8d7da"
    return ""

display(stability_df[["GEOID", "Tract",
                       "Rank_Baseline", "Rank_Transit-Heavy", "Rank_Residential",
                       "In N scenarios", "Stability"]].style
        .format({"Rank_Baseline": "{:.0f}", "Rank_Transit-Heavy": "{:.0f}", "Rank_Residential": "{:.0f}"})
        .hide(axis="index")
        .map(highlight_stability, subset=["Stability"])
)

In [ ]:
# =====================================================================
# Part B — Competitor weight sensitivity (3 settings)
# =====================================================================

COMP_WEIGHTS = {
    "Ignore\n(cw=0.0)": 0.0,
    "Half\n(cw=0.5)": 0.5,
    "Full\n(cw=1.0)": 1.0,
}

comp_results = {}
for name, cw in COMP_WEIGHTS.items():
    scored = compute_lfs(
        lfs_input,
        demand_cols=DEMAND_COLS,
        demand_weights=DEMAND_WEIGHTS,  # use baseline demand weights
        brand_col="n_starbucks",
        competitor_col="n_competitor_cafes",
        competitor_weight=cw,
    )
    comp_results[name] = scored[["GEOID", "lfs"]].copy()

# Build comparison
comp_compare = lfs_input[["GEOID", "NAMELSAD"]].copy()
for name, res in comp_results.items():
    comp_compare = comp_compare.merge(
        res.rename(columns={"lfs": name}), on="GEOID", how="left"
    )

# Spearman correlation
comp_names = list(COMP_WEIGHTS.keys())
comp_short = ["Ignore (0.0)", "Half (0.5)", "Full (1.0)"]
n_cw = len(comp_names)
rho_comp = np.ones((n_cw, n_cw))
for i in range(n_cw):
    for j in range(i + 1, n_cw):
        rho, _ = spearmanr(comp_compare[comp_names[i]], comp_compare[comp_names[j]])
        rho_comp[i, j] = rho
        rho_comp[j, i] = rho

fig8 = go.Figure(data=go.Heatmap(
    z=rho_comp,
    x=comp_short, y=comp_short,
    text=rho_comp.round(3),
    texttemplate="%{text}",
    textfont={"size": 15},
    colorscale="Oranges", zmin=0.5, zmax=1.0,
))
fig8.update_layout(
    title="Fig. 8 — Spearman Rank Correlation Across Competitor Weight Settings",
    height=400, width=500,
    template="plotly_white",
)
fig8.show()

min_rho_comp = rho_comp[np.triu_indices(n_cw, k=1)].min()
print(f"Minimum pairwise Spearman ρ = {min_rho_comp:.3f}")
if min_rho_comp >= 0.8:
    print("✓ Rankings are ROBUST to competitor weight changes (all ρ ≥ 0.8).")
elif min_rho_comp >= 0.6:
    print("△ Rankings are MODERATELY sensitive to competitor weight changes.")
else:
    print("✗ Rankings are HIGHLY sensitive — competitor weight choice matters.")

In [ ]:
# ----- Competitor weight: top 10 stability check -----
comp_top_sets = {}
for name in comp_names:
    has_d = comp_compare[comp_compare[name] > 0]
    comp_top_sets[name] = set(has_d.nlargest(TOP_N, name)["GEOID"].values)

comp_stable = set.intersection(*comp_top_sets.values())
comp_all = set.union(*comp_top_sets.values())
comp_one = {g for g in comp_all if sum(g in s for s in comp_top_sets.values()) == 1}

print(f"Competitor weight sensitivity — Top {TOP_N} under-served:")
print(f"  ★ Stable (in all 3 settings):  {len(comp_stable)}")
print(f"  Partial (in 2 settings):        {len(comp_all) - len(comp_stable) - len(comp_one)}")
print(f"  Unstable (in 1 setting only):   {len(comp_one)}")

# =====================================================================
# Combined robustness summary
# =====================================================================
# Tracts that are stable across BOTH demand AND competitor sensitivity
both_stable = stable & comp_stable

print()
print("=" * 70)
print(f"ROBUST CANDIDATES — Stable across ALL 6 tests")
print(f"  (3 demand scenarios × 2 + 3 competitor settings)")
print("=" * 70)

if both_stable:
    robust_df = compare[compare["GEOID"].isin(both_stable)][["GEOID", "NAMELSAD"]].copy()
    # Add baseline LFS for reference
    robust_df = robust_df.merge(
        tracts_scored[["GEOID", "lfs", "n_starbucks", "n_competitor_cafes", "demand_proxy_index"]],
        on="GEOID", how="left"
    )
    robust_df.columns = ["GEOID", "Tract", "LFS (Baseline)", "Starbucks", "Competitors", "DPI"]
    display(robust_df.style.format({"LFS (Baseline)": "{:.4f}", "DPI": "{:.3f}"}).hide(axis="index"))
else:
    print("No tracts are stable across ALL tests — rankings are sensitive to assumptions.")

print(f"\nOverall robustness assessment:")
print(f"  Demand weight sensitivity:     min ρ = {min_rho:.3f} {'✓' if min_rho >= 0.8 else '△' if min_rho >= 0.6 else '✗'}")
print(f"  Competitor weight sensitivity:  min ρ = {min_rho_comp:.3f} {'✓' if min_rho_comp >= 0.8 else '△' if min_rho_comp >= 0.6 else '✗'}")
print(f"  Stable across demand scenarios: {len(stable)}/{TOP_N} tracts")
print(f"  Stable across comp. settings:   {len(comp_stable)}/{TOP_N} tracts")
print(f"  Stable across BOTH:             {len(both_stable)} tracts")

### Geographically Weighted Regression: Do Demand Drivers Vary by Location?

The LFS above uses **global weights** (e.g., 50% MTA, 25% pedestrian, 25% population). But do these relationships hold everywhere? In Midtown, subway ridership may dominate; in Upper Manhattan, population density may matter more.

**GWR (Geographically Weighted Regression)** fits a separate regression at each location, weighting nearby observations more heavily. The result: a map of **local coefficients** showing where each variable matters most.

- **Dependent variable:** Starbucks store count per tract
- **Predictors:** MTA ridership, pedestrian count (IDW), population density (all normalized)

In [ ]:
# ----- GWR: Geographically Weighted Regression -----
try:
    from mgwr.gwr import GWR
    from mgwr.sel_bw import Sel_BW
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'mgwr'])
    from mgwr.gwr import GWR
    from mgwr.sel_bw import Sel_BW

import statsmodels.api as sm

# Prepare data — use tracts_scored which has all variables
gwr_data = tracts_scored.copy()
gwr_data = gwr_data[gwr_data.geometry.is_valid].reset_index(drop=True)

# Coordinates (centroids in meters)
centroids = gwr_data.geometry.centroid
coords = np.column_stack([
    centroids.x.values * M_PER_DEG_LON,
    centroids.y.values * M_PER_DEG_LAT,
])

# Dependent variable
y = gwr_data["n_starbucks"].values.reshape(-1, 1).astype(float)

# Predictors (normalized columns from compute_lfs)
pred_cols = ["mta_ridership_norm", "ped_count_idw_norm", "tract_pop_density_norm"]
X = gwr_data[pred_cols].values.astype(float)

# --- OLS baseline ---
X_ols = sm.add_constant(X)
ols_model = sm.OLS(y, X_ols).fit()
print("OLS Baseline:")
print(f"  R² = {ols_model.rsquared:.3f}, Adj R² = {ols_model.rsquared_adj:.3f}")
print(f"  AICc = {ols_model.aic:.1f}")
for name, coef, pval in zip(["const"] + pred_cols, ols_model.params, ols_model.pvalues):
    sig = "*" if pval < 0.05 else ""
    print(f"    {name:30s}: β = {coef:.3f}  (p = {pval:.3f}){sig}")

# --- GWR ---
print("\nFitting GWR (bandwidth selection may take a moment)...")
bw_selector = Sel_BW(coords, y, X, kernel='bisquare', fixed=False)
bw = bw_selector.search(criterion='AICc')
print(f"  Optimal bandwidth: {bw:.0f} nearest neighbors")

gwr_model = GWR(coords, y, X, bw=bw, kernel='bisquare', fixed=False).fit()

print(f"\nGWR Results:")
print(f"  R² = {gwr_model.R2:.3f}")
print(f"  Adj R² = {gwr_model.adj_R2:.3f}")
print(f"  AICc = {gwr_model.aicc:.1f}")
print(f"  Improvement over OLS: ΔR² = {gwr_model.R2 - ols_model.rsquared:+.3f}")

# Store local coefficients
for j, name in enumerate(pred_cols):
    gwr_data[f"gwr_coef_{name}"] = gwr_model.params[:, j+1]  # +1 to skip intercept
gwr_data["gwr_local_r2"] = gwr_model.localR2.flatten()

In [ ]:
# ----- GWR coefficient maps -----
gwr_maps = [
    ("gwr_coef_mta_ridership_norm", "β: MTA Ridership", "RdYlBu_r"),
    ("gwr_coef_ped_count_idw_norm", "β: Pedestrian Count", "RdYlBu_r"),
    ("gwr_coef_tract_pop_density_norm", "β: Population Density", "RdYlBu_r"),
    ("gwr_local_r2", "Local R²", "YlOrRd"),
]

for col, label, cmap in gwr_maps:
    m = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")
    folium.Choropleth(
        geo_data=gwr_data.to_json(),
        data=gwr_data,
        columns=["GEOID", col],
        key_on="feature.properties.GEOID",
        fill_color=cmap,
        fill_opacity=0.7,
        line_opacity=0.3,
        legend_name=label,
    ).add_to(m)
    print(f"\n{label}:")
    print(f"  Range: {gwr_data[col].min():.3f} – {gwr_data[col].max():.3f}")
    print(f"  Mean:  {gwr_data[col].mean():.3f}")
    show_map(m)

**GWR findings:**

- **GWR outperforms OLS** — the spatially varying model captures local demand patterns that global weights miss
- **MTA ridership coefficient varies by area** — strongest in Midtown where transit drives foot traffic, weaker in residential areas where population density matters more
- **Local R²** shows where the model fits well vs poorly — low R² areas may have unmeasured demand drivers (e.g., tourist attractions, office density)
- This validates the intuition behind the sensitivity analysis: **no single weight scheme is optimal everywhere**

### Huff Model: Estimating Store-Level Market Share

The **Huff model** (1963) estimates the probability that a consumer at location $j$ patronizes store $i$:

$$P_{ij} = \frac{A_i \cdot d_{ij}^{-\beta}}{\sum_{k=1}^{n} A_k \cdot d_{kj}^{-\beta}}$$

where $A_i$ is store attractiveness (we use nearest station ridership), $d_{ij}$ is distance, and $\beta$ controls distance decay.

This extends the LFS by modeling **competition explicitly** — a high-demand tract near many stores splits its demand among them.

In [ ]:
# ----- Huff Model: Store-level market share estimation -----

# Store locations and attractiveness
stores_df = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")
store_xy = np.column_stack([
    stores_df["lon"].values * M_PER_DEG_LON,
    stores_df["lat"].values * M_PER_DEG_LAT,
])

# Attractiveness: nearest station ridership (proxy for foot traffic potential)
attractiveness = stores_df["mta_avg_daily_ridership"].fillna(0).values + 1  # +1 to avoid zero

# Tract centroids (demand origins)
tract_centroids = gwr_data.geometry.centroid
tract_xy = np.column_stack([
    tract_centroids.x.values * M_PER_DEG_LON,
    tract_centroids.y.values * M_PER_DEG_LAT,
])

# Distance matrix: tracts × stores
from scipy.spatial.distance import cdist
dist_matrix = cdist(tract_xy, store_xy)  # in meters
dist_matrix = np.maximum(dist_matrix, 50)  # floor at 50m to avoid division issues

# Huff probability matrix
BETA = 2.0  # distance decay exponent
utility = attractiveness[np.newaxis, :] * dist_matrix ** (-BETA)
huff_prob = utility / utility.sum(axis=1, keepdims=True)

# Tract-level demand (use DPI as proxy)
tract_demand = gwr_data["demand_proxy_index"].values

# Store-level expected demand share
store_demand = (huff_prob * tract_demand[:, np.newaxis]).sum(axis=0)
stores_df["huff_demand"] = store_demand

# Catchment area: for each tract, which store captures the most probability?
gwr_data["huff_dominant_store"] = huff_prob.argmax(axis=1)
gwr_data["huff_max_prob"] = huff_prob.max(axis=1)

# Effective competition: how many stores share >5% of each tract's demand?
gwr_data["huff_n_competitors"] = (huff_prob > 0.05).sum(axis=1)

print(f"Huff Model Summary (β = {BETA}):")
print(f"  Stores: {len(stores_df)}")
print(f"  Tracts: {len(gwr_data)}")
print(f"  Max probability (single store capturing a tract): {gwr_data['huff_max_prob'].max():.2%}")
print(f"  Median effective competitors per tract (>5%): {gwr_data['huff_n_competitors'].median():.0f}")
print(f"\n  Store demand range: {store_demand.min():.4f} – {store_demand.max():.4f}")
print(f"  Top 5 stores by Huff demand share:")
top5 = stores_df.nlargest(5, 'huff_demand')
for _, row in top5.iterrows():
    print(f"    {str(row.get('addr_street', 'N/A')):40s}  demand = {row['huff_demand']:.4f}")

In [ ]:
# ----- Huff model visualization: effective competition map -----
m_huff = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=gwr_data.to_json(),
    data=gwr_data,
    columns=["GEOID", "huff_n_competitors"],
    key_on="feature.properties.GEOID",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Effective Competitors (Huff >5%)",
).add_to(m_huff)

# Add store markers sized by Huff demand
demand_norm = stores_df["huff_demand"] / stores_df["huff_demand"].max()
for _, row in stores_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=2 + demand_norm[row.name] * 8,
        color="#D62828",
        fill=True,
        fill_opacity=0.7,
        weight=0.5,
    ).add_to(m_huff)

show_map(m_huff)

# Scatter: Huff demand vs LFS
# Map each store to its tract's LFS
store_tracts = gpd.GeoDataFrame(
    stores_df, geometry=gpd.points_from_xy(stores_df.lon, stores_df.lat), crs="EPSG:4326"
)
store_tract_join = gpd.sjoin(store_tracts, gwr_data[["GEOID", "geometry", "lfs"]], how="left", predicate="within")

r_huff_lfs = store_tract_join["huff_demand"].corr(store_tract_join["lfs"])
print(f"\nCorrelation: Huff demand vs LFS = {r_huff_lfs:.3f}")
print(f"→ {'Strong' if abs(r_huff_lfs) > 0.5 else 'Moderate' if abs(r_huff_lfs) > 0.3 else 'Weak'} agreement between models")

**Huff Model findings:**

- The Huff model complements the LFS by explicitly modeling **competitive interactions** — a high-DPI tract with many stores divides its demand
- **Effective competition** varies from 1-2 stores in outer Manhattan to 10+ in Midtown, quantifying the cannibalization pressure
- Stores with highest Huff demand share tend to be near high-ridership stations with fewer nearby competitors — the spatial sweet spot
- The correlation between Huff demand and LFS validates that both approaches identify similar high-opportunity locations

> **Template note:** The Huff model works for any retail network. Replace `attractiveness` with your own store features (square footage, brand strength, etc.) and adjust `β` based on your industry's distance sensitivity.

## Section 5 — Backtest: Do Closed Stores Validate the Score?

Can we check whether the LFS "would have predicted" past store closures?

We compare two snapshots:
- **2017**: [Kaggle Starbucks directory.csv](https://www.kaggle.com/datasets/starbucks/store-locations) — 232 stores in Manhattan (25,600 worldwide)
- **2026**: Our OSM dataset — 171 stores

That's roughly **60 stores gone** in 9 years. If the LFS model is meaningful, closed stores should cluster in **over-served tracts** (low LFS), not under-served ones.

### Matching strategy

The 2017 Kaggle data has coordinates rounded to **2 decimal places** (~1.1 km precision at NYC's latitude). We cannot match stores building-by-building, so we use **proximity matching**: for each 2017 store, find the nearest 2026 store. If no 2026 store exists within 500m, classify it as "likely closed." We then compare the LFS of tracts containing likely-closed stores against tracts with current stores.

> **Reuse tip:** Replace the 2017 CSV with any historical store list to backtest your own LFS model.

In [ ]:
# ----- Load 2017 Kaggle data and filter to Manhattan -----
KAGGLE_2017_PATH = None
_candidates = [
    Path("/kaggle/input/datasets/organizations/starbucks/store-locations/directory.csv"),
    Path("/kaggle/input/store-locations/directory.csv"),
    Path("/kaggle/input/datasets/starbucks/store-locations/directory.csv"),
    DATA_DIR / "directory.csv",
    Path("../../data/raw/kaggle_starbucks/directory.csv"),  # local path
]
# Also search recursively as fallback
if Path("/kaggle/input").exists():
    found = list(Path("/kaggle/input").rglob("directory.csv"))
    _candidates = found + _candidates

for _p in _candidates:
    if _p.exists():
        KAGGLE_2017_PATH = _p
        break

if KAGGLE_2017_PATH is None:
    print("⚠ 2017 Kaggle directory.csv not found. Skipping backtest.")
    print("  To run this section, download from:")
    print("  https://www.kaggle.com/datasets/starbucks/store-locations")
    dir_2017 = None
else:
    dir_all = pd.read_csv(KAGGLE_2017_PATH)
    # Filter to Manhattan (New York City, NY, bounding box)
    dir_2017 = dir_all[
        (dir_all["City"].str.lower().isin(["new york", "new york city"])) &
        (dir_all["State/Province"] == "NY") &
        (dir_all["Longitude"] >= -74.03) & (dir_all["Longitude"] <= -73.90) &
        (dir_all["Latitude"] >= 40.70) & (dir_all["Latitude"] <= 40.88)
    ].copy()
    print(f"2017 Kaggle Manhattan stores: {len(dir_2017)}")
    print(f"2026 OSM Manhattan stores:    {len(df)}")
    print(f"Net change: {len(df) - len(dir_2017):+d} stores")
    print(f"\nCoordinate precision (2017):")
    print(f"  Longitude: {dir_2017['Longitude'].apply(lambda x: len(str(x).split('.')[-1])).median():.0f} decimal places")
    print(f"  Latitude:  {dir_2017['Latitude'].apply(lambda x: len(str(x).split('.')[-1])).median():.0f} decimal places")
    print(f"  ≈ {111_320 * 0.01:.0f}m precision (1 deg ≈ 111km at this latitude)")

In [ ]:
# ----- Match 2017 → 2026 stores by proximity -----
if dir_2017 is not None:
    # Strategy: For each 2017 store, find the nearest 2026 store.
    # If nearest match > 500m, classify as "likely closed"
    # (500m threshold accounts for ~1km coordinate imprecision in 2017 data)

    coords_2017 = np.column_stack([
        dir_2017["Latitude"].values * M_PER_DEG_LAT,
        dir_2017["Longitude"].values * M_PER_DEG_LON,
    ])
    coords_2026 = np.column_stack([
        df["lat"].values * M_PER_DEG_LAT,
        df["lon"].values * M_PER_DEG_LON,
    ])

    tree_2026 = cKDTree(coords_2026)
    dists, idxs = tree_2026.query(coords_2017)

    dir_2017 = dir_2017.copy()
    dir_2017["nearest_2026_dist_m"] = dists
    dir_2017["nearest_2026_idx"] = idxs

    MATCH_THRESHOLD_M = 500
    dir_2017["matched"] = dir_2017["nearest_2026_dist_m"] <= MATCH_THRESHOLD_M

    n_matched = dir_2017["matched"].sum()
    n_unmatched = (~dir_2017["matched"]).sum()
    print(f"Proximity matching (threshold = {MATCH_THRESHOLD_M}m):")
    print(f"  Matched (likely still open): {n_matched}")
    print(f"  Unmatched (likely closed):   {n_unmatched}")
    print(f"  Match rate: {n_matched/len(dir_2017)*100:.1f}%")

    # Assign likely-closed stores to Census Tracts via spatial join
    closed_2017 = dir_2017[~dir_2017["matched"]].copy()
    closed_gdf = gpd.GeoDataFrame(
        closed_2017,
        geometry=gpd.points_from_xy(closed_2017["Longitude"], closed_2017["Latitude"]),
        crs="EPSG:4326",
    )
    closed_by_tract = gpd.sjoin(closed_gdf, tracts_gdf[["GEOID", "geometry"]], how="left", predicate="within")

    # Merge LFS into closed stores
    closed_by_tract = closed_by_tract.merge(
        tracts_scored[["GEOID", "lfs", "demand_proxy_index", "supply_index"]],
        on="GEOID", how="left"
    )

    # Compare LFS: closed stores' tracts vs all tracts with stores
    tracts_with_stores = tracts_scored[tracts_scored["n_starbucks"] > 0]

    closed_lfs = closed_by_tract["lfs"].dropna()
    surviving_lfs = tracts_with_stores["lfs"]

    print(f"\n--- LFS Comparison ---")
    print(f"Closed stores' tracts ({len(closed_lfs)} stores):")
    print(f"  LFS median = {closed_lfs.median():.4f}")
    print(f"  LFS mean   = {closed_lfs.mean():.4f}")
    print(f"Current store tracts ({len(surviving_lfs)} tracts):")
    print(f"  LFS median = {surviving_lfs.median():.4f}")
    print(f"  LFS mean   = {surviving_lfs.mean():.4f}")

    # Mann-Whitney U test
    from scipy.stats import mannwhitneyu
    if len(closed_lfs) >= 5 and len(surviving_lfs) >= 5:
        stat, p = mannwhitneyu(closed_lfs, surviving_lfs, alternative="less")
        print(f"\nMann-Whitney U test (H₁: closed-store LFS < current-store LFS):")
        print(f"  U = {stat:.1f}, p = {p:.4f}")
        if p < 0.05:
            print("  ✓ Significant — closed stores cluster in lower-LFS (over-served) tracts.")
        elif p < 0.10:
            print("  △ Marginally significant (p < 0.10) — directional evidence only.")
        else:
            print("  ✗ Not significant — no clear LFS difference between closed and current stores.")
    else:
        print(f"\n  Sample too small for statistical test (closed={len(closed_lfs)}, surviving={len(surviving_lfs)}).")

    print(f"\n⚠ Caveat: 2017 coordinates have ~{MATCH_THRESHOLD_M}m precision.")
    print(f"  Some 'closures' may be misclassified due to coordinate rounding.")
    print(f"  Treat results as directional evidence, not proof.")
else:
    closed_by_tract = None
    print("Backtest skipped — 2017 data not available.")

In [ ]:
# ----- Fig. 9 & 10: Backtest visualization -----
if dir_2017 is not None and closed_by_tract is not None and len(closed_by_tract) > 0:
    # Build comparison dataframe for plotting
    closed_plot = closed_by_tract[["lfs"]].dropna().copy()
    closed_plot["group"] = "Likely Closed\n(2017→2026)"

    surviving_plot = tracts_with_stores[["lfs"]].copy()
    surviving_plot["group"] = "Current Stores\n(2026)"

    plot_data = pd.concat([closed_plot, surviving_plot], ignore_index=True)

    fig9 = px.box(
        plot_data, x="group", y="lfs", color="group",
        color_discrete_map={
            "Likely Closed\n(2017→2026)": "#d62728",
            "Current Stores\n(2026)": "#2ca02c",
        },
        labels={"group": "", "lfs": "LFS (Baseline)"},
        title="Fig. 9 — LFS of Likely-Closed vs Current Store Tracts",
        template="plotly_white", height=400,
    )
    fig9.update_layout(showlegend=False)
    fig9.show()

    # Show distance distribution of unmatched stores
    fig10 = px.histogram(
        dir_2017, x="nearest_2026_dist_m", nbins=50,
        color="matched",
        color_discrete_map={True: "#2ca02c", False: "#d62728"},
        labels={"nearest_2026_dist_m": "Distance to nearest 2026 store (m)",
                "matched": "Matched"},
        title="Fig. 10 — Distance from Each 2017 Store to Its Nearest 2026 Counterpart",
        template="plotly_white", height=350,
    )
    fig10.add_vline(x=MATCH_THRESHOLD_M, line_dash="dash", line_color="gray",
                    annotation_text=f"threshold = {MATCH_THRESHOLD_M}m")
    fig10.show()
else:
    print("Backtest visualization skipped.")

## Section 6 — Walk-Distance Demo with OSMnx

So far we've used **straight-line distances**. In reality, pedestrians walk along streets, not as-the-crow-flies. [OSMnx](https://github.com/gboeing/osmnx) lets us compute actual walking distances on the street network.

Below is a **2-store demo** showing the 5-minute walk isochrone (400m network distance) from selected Starbucks locations. This gives a more realistic picture of each store's "service area."

**Why only 2 stores?** Computing walk-distance polygons for all 171 stores requires downloading Manhattan's full street graph and running shortest-path algorithms for each store — about 10–15 minutes total. To keep this notebook fast, we demonstrate the technique on 2 stores and provide the full-batch code as a template.

> **Reuse tip:** Change `demo_stores` to any list of (lat, lon) pairs to compute isochrones for your own locations.

In [ ]:
# ----- OSMnx 5-minute walk isochrone demo (2 stores) -----
import osmnx as ox
import networkx as nx

WALK_SPEED_M_PER_MIN = 80   # ~4.8 km/h
WALK_TIME_MIN = 5
WALK_DIST_M = WALK_SPEED_M_PER_MIN * WALK_TIME_MIN  # 400m

# Pick 2 demo stores: one in Midtown (high density), one in Upper West Side
demo_stores = df.sort_values("mta_avg_daily_ridership", ascending=False).head(20)
store_midtown = demo_stores.iloc[0]  # highest ridership area
store_uptown = df[df["lat"] > 40.78].sort_values("mta_avg_daily_ridership", ascending=False).iloc[0]

demo_list = [
    {"name": store_midtown.get("name", "Midtown Store"), "lat": store_midtown["lat"], "lon": store_midtown["lon"]},
    {"name": store_uptown.get("name", "Uptown Store"), "lat": store_uptown["lat"], "lon": store_uptown["lon"]},
]

print(f"Demo stores:")
for s in demo_list:
    print(f"  {s['name']} ({s['lat']:.5f}, {s['lon']:.5f})")

# Download walking network around demo stores (small bbox to keep it fast)
lats = [s["lat"] for s in demo_list]
lons = [s["lon"] for s in demo_list]
center_pt = (np.mean(lats), np.mean(lons))

# Use a bounding box that covers both stores + 800m buffer
print("\nDownloading Manhattan walking network (this may take a moment)...")
G = ox.graph_from_point(center_pt, dist=3000, network_type="walk")
print(f"  Nodes: {G.number_of_nodes():,}, Edges: {G.number_of_edges():,}")

# Add edge lengths if not present
G = ox.distance.add_edge_lengths(G)

In [ ]:
# ----- Compute isochrones and visualize on folium -----
from shapely.geometry import MultiPoint

def compute_isochrone(G, lat, lon, max_dist_m=400):
    """Compute the walk-distance isochrone polygon from a point."""
    center_node = ox.nearest_nodes(G, lon, lat)
    subgraph = nx.ego_graph(G, center_node, radius=max_dist_m, distance="length")
    reachable_nodes = list(subgraph.nodes)
    node_coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in reachable_nodes]
    if len(node_coords) < 3:
        return None
    points = MultiPoint(node_coords)
    return points.convex_hull

# ----- Fig. 11: Isochrone map -----
m11 = folium.Map(location=center_pt, zoom_start=14, tiles="CartoDB positron")

colors = ["#1f77b4", "#ff7f0e"]
for i, store in enumerate(demo_list):
    iso = compute_isochrone(G, store["lat"], store["lon"], WALK_DIST_M)
    if iso is not None:
        folium.GeoJson(
            iso.__geo_interface__,
            style_function=lambda x, c=colors[i]: {
                "fillColor": c, "color": c, "weight": 2,
                "fillOpacity": 0.25,
            },
            tooltip=f"{store['name']}: {WALK_TIME_MIN}-min walk ({WALK_DIST_M}m)",
        ).add_to(m11)

    folium.Marker(
        location=[store["lat"], store["lon"]],
        icon=folium.Icon(color="green", icon="coffee", prefix="fa"),
        popup=f"<b>{store['name']}</b><br>{WALK_TIME_MIN}-min walk isochrone",
    ).add_to(m11)

# Nearby Starbucks for context
for _, row in df.iterrows():
    if abs(row["lat"] - center_pt[0]) < 0.02 and abs(row["lon"] - center_pt[1]) < 0.02:
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=4, color="#00704A", fill=True, fill_opacity=0.6,
            popup=row.get("name", "Starbucks"),
        ).add_to(m11)

m11.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;top:10px;left:60px;z-index:9999;"
    "font-size:14px;font-weight:bold;background:white;padding:6px 12px;"
    "border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,0.3);'>"
    f"Fig. 11 — {WALK_TIME_MIN}-Minute Walk Isochrones (OSMnx network distance)</div>"
))

show_map(m11)

print(f"\nIsochrone demo complete.")
print(f"  Walk distance: {WALK_DIST_M}m ({WALK_TIME_MIN} min at {WALK_SPEED_M_PER_MIN} m/min)")
print(f"  Stores shown: {len(demo_list)}")
print(f"  To run for all 171 stores, use compute_isochrone() in a loop")
print(f"  with a city-wide graph: G = ox.graph_from_place('Manhattan, NY', network_type='walk')")

## Section 7 — Limitations & What's Next

### What this model can and cannot do

| Limitation | Impact | Mitigation |
|---|---|---|
| **No actual sales data** | Demand weights are assumptions, not calibrated parameters | Sensitivity analysis (Section 4) showed rankings are robust to weight changes (Spearman ρ ≥ 0.89) |
| **Pedestrian counts: 36 locations only** | Most tracts have no direct foot-traffic measurement | Used spatial join; tracts without counters default to zero. A city-wide sensor network would improve coverage |
| **OSM coverage ≈ 85%** | Some Starbucks may be missing from the 2026 dataset | Cross-referenced with Google Maps during data collection. 171 stores is consistent with Starbucks' own filings |
| **Census Tract granularity** | Tracts average ~0.1 km² but demand can vary block-by-block | Point-level nearest-neighbor analysis (Notebook A) complements tract-level scoring |
| **Static snapshot** | Captures one moment — ignores seasonal variation, store openings/closures in progress | Backtest (Section 5) provided partial temporal validation with 2017 data |
| **Straight-line ≠ walking distance** | Cannibalization radius may be over/underestimated | OSMnx demo (Section 6) showed how to compute actual walk-distance isochrones |
| **2017 backtest coordinates: 2 decimal places** | ~1 km precision limits store-level matching | Used 500m proximity threshold and acknowledged results as directional evidence |

### What we found

1. **LFS identifies under-served and over-served tracts.** The top expansion candidates are tracts with subway ridership and population density but few existing cafés.
2. **Rankings are reasonably robust.** Spearman rank correlations across demand weight scenarios (ρ = 0.89–0.97) and competitor weight settings (ρ = 0.94–0.99) are all above 0.8.
3. **Backtest supports the model.** Likely-closed stores (2017→2026) cluster in lower-LFS tracts (Mann-Whitney p < 0.001), suggesting the score captures something real — though coordinate precision limits the strength of this conclusion.

### The Series

| Notebook | Topic | Link |
|----------|-------|------|
| **Theme 0** — Manhattan Cafe Wars | EDA, competitor mapping, OSMnx demo | [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| **Theme 1** — 10-K NLP Analysis | 30 years of keyword trends, LDA topics, NLP × store count | [Starbucks 10-K NLP](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| **Theme 2A** — Spatial Clustering | Moran's I, LISA, Ripley's K | [Starbucks Spatial Clustering](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| **Theme 2B** — Location Fitness | Demand-supply gap scoring, sensitivity analysis, backtest | You are here |

**Datasets:**
- [Manhattan Café Wars: Starbucks & Subway](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars) — spatial data (Theme 0, 2A, 2B)
- [Starbucks 30-Year 10-K NLP Corpus](https://www.kaggle.com/datasets/shiratoriseto/starbucks-30year-nlp-corpus) — NLP data (Theme 1)

---

*Built with open data: OpenStreetMap (ODbL), NYC Open Data, MTA, US Census Bureau, NYC DOT.*
*Analysis code is MIT-licensed — fork, adapt, and apply to your own city.*